In [1]:
# Cell 1: Import necessary libraries and environment setup
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score

# Configure device and check GPU specifications
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("-" * 50)
print(f"🚀 Execution Device: {device}")

if torch.cuda.is_available():
    # Let's flex the hardware for your teammates!
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert to GB
    print(f"🔥 Hardware Model: {gpu_name}")
    print(f"📟 Dedicated VRAM: {gpu_memory:.2f} GB")
    print(f"✨ CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Warning: Running on CPU. Performance might be limited.")
print("-" * 50)

# Global setting: Optimization for large-scale audio processing
# Note: We are using librosa for loading to bypass torchaudio/torchcodec version conflicts
import librosa 
print("✅ Environment initialized. Ready for FinQuest 2026.")

--------------------------------------------------
🚀 Execution Device: cuda
🔥 Hardware Model: NVIDIA GeForce RTX 5080
📟 Dedicated VRAM: 15.46 GB
✨ CUDA Capability: (12, 0)
--------------------------------------------------
✅ Environment initialized. Ready for FinQuest 2026.


In [2]:
# Cell 2: Data Preparation and Dataset Class Definition
import os
import pandas as pd
import librosa
import torch
import numpy as np
import random
from torch.utils.data import Dataset
from transformers import AutoProcessor
from audiomentations import Compose, AddGaussianNoise, Shift, Gain, PitchShift, TimeMask

# 1. 🛠️ Initialize the Wav2Vec2 Processor
# This handles the feature extraction (normalization, 16kHz alignment, etc.)
MODEL_ID = "facebook/wav2vec2-base"
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 2. 📂 Load the Master Protocol
protocol_path = "./Protocol/master_metadata.csv"
if not os.path.exists(protocol_path):
    raise FileNotFoundError(f"Could not find {protocol_path}. Please check the path!")

master_df = pd.read_csv(protocol_path)

# 3. 🏷️ Create Label Mapping (7 classes)
label_list = sorted(master_df['emotion'].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# 4. ✂️ Map splits from the Protocol
train_df = master_df[master_df['split'] == 'train'].reset_index(drop=True)
dev_df   = master_df[master_df['split'] == 'dev'].reset_index(drop=True)
test_df  = master_df[master_df['split'] == 'test'].reset_index(drop=True)

# --- Dataset Class Definition ---
class VoxSentinelDataset(Dataset):
    def __init__(self, dataframe, processor, label_map, max_length=64000, augment=False):
        self.df = dataframe
        self.processor = processor
        self.label_map = label_map
        self.max_length = max_length
        self.augment = augment
        self.clean_source_keywords = ['tess', 'ravdess', 'crema']
        
        if self.augment:
            self.augmenter = Compose([
                Gain(min_gain_db=-10, max_gain_db=5, p=0.5),
                PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
                AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.3),
                TimeMask(min_band_part=0.05, max_band_part=0.1, p=0.2),
                Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path, emotion, dataset_name = row['path'], row['emotion'], row['dataset']
        label = self.label_map[emotion]

        try:
            # Load 16kHz Mono
            waveform, _ = librosa.load(path, sr=16000, mono=True)
            waveform = np.nan_to_num(waveform)
            
            # Asymmetric Augmentation
            if self.augment:
                is_clean = any(k in dataset_name.lower() for k in self.clean_source_keywords)
                if is_clean:
                    waveform = self.augmenter(samples=waveform.astype(np.float32), sample_rate=16000)
                else:
                    if random.random() < 0.5: waveform *= random.uniform(0.8, 1.2)
                waveform = np.nan_to_num(waveform)

            # Cyclic Tiling (Filling the 4s window)
            if len(waveform) >= self.max_length:
                start = random.randint(0, len(waveform) - self.max_length) if self.augment else (len(waveform) - self.max_length) // 2
                waveform = waveform[start : start + self.max_length]
            else:
                waveform = np.tile(waveform, (self.max_length // len(waveform)) + 1)[:self.max_length]

            # Use processor for normalization
            inputs = self.processor(torch.from_numpy(waveform.copy()), sampling_rate=16000, return_tensors="pt")
            input_values = torch.clamp(inputs.input_values.squeeze(0), -10.0, 10.0)

            return {
                "input_values": input_values,
                "attention_mask": torch.ones(self.max_length, dtype=torch.long),
                "label": torch.tensor(label, dtype=torch.long)
            }
        except Exception:
            return self.__getitem__((idx + 1) % len(self.df))

# --- Initialize Loaders ---
train_dataset = VoxSentinelDataset(train_df, processor, label2id, augment=True)
dev_dataset   = VoxSentinelDataset(dev_df, processor, label2id, augment=False)

print("-" * 60)
print(f"✅ DataLoaders ready for Training Loop.")
print(f"🔹 Emotion Map: {label2id}")
print(f"🔹 Train Samples: {len(train_df)} | Dev: {len(dev_df)}")
print("-" * 60)

------------------------------------------------------------
✅ DataLoaders ready for Training Loop.
🔹 Emotion Map: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
🔹 Train Samples: 11074 | Dev: 1326
------------------------------------------------------------


In [3]:
# Cell 3: Model Architecture with Fixed Multi-Scale Attentive Pooling
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model

class MultiScaleAttentivePooling(nn.Module):
    """
    Evolved Pooling: Captures Weighted Mean and Weighted Std.
    Fixed: Resizes the 64,000-length attention mask to match 
    the ~199-length Transformer output.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x, attention_mask=None):
        # x shape: [batch, seq_len, 768] (seq_len is usually 199 for 4s audio)
        attn_logits = self.attention(x) # [batch, seq_len, 1]
        
        if attention_mask is not None:
            # 🚀 DOWNSAMPLING MASK:
            # Wav2Vec2 CNN downsamples 16kHz audio by 320x.
            # We use nearest interpolation to align the 64000 mask to seq_len (199).
            # [batch, 64000] -> [batch, 1, 64000]
            mask = attention_mask.unsqueeze(1).float()
            # [batch, 1, 64000] -> [batch, 1, seq_len]
            mask = nn.functional.interpolate(mask, size=x.shape[1], mode='nearest').squeeze(1)
            # [batch, seq_len] -> [batch, seq_len, 1]
            mask = mask.unsqueeze(-1)
            
            # Apply Mask: Set padded regions to highly negative values
            attn_logits = attn_logits.masked_fill(mask == 0, -1e9)
            
        attn_weights = torch.softmax(attn_logits, dim=1) # [batch, seq_len, 1]
        
        # 1. Calculate Weighted Mean (mu)
        mu = torch.sum(attn_weights * x, dim=1) # [batch, 768]
        
        # 2. Calculate Weighted Standard Deviation (sigma)
        delta = x - mu.unsqueeze(1)
        var = torch.sum(attn_weights * (delta ** 2), dim=1)
        std = torch.sqrt(torch.clamp(var, min=1e-9)) # [batch, 768]

        # Concatenate Mean and Std -> 1536 dimensions
        return torch.cat([mu, std], dim=-1)

class VoxSentinelEmotionModel(nn.Module):
    """
    VoxSentinel Main Model: Wav2Vec2-Base + Multi-Scale Attentive Pooling.
    """
    def __init__(self, num_labels):
        super().__init__()
        # Load the pre-trained feature extractor and transformer
        self.backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        
        # Multi-Scale Pooling (Mu + Sigma)
        self.pooling = MultiScaleAttentivePooling(768)
        
        # Classification MLP
        self.classifier = nn.Sequential(
            nn.Linear(1536, 512), 
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_values, attention_mask=None):
        # 1. Feature Extraction via Wav2Vec2
        # outputs.last_hidden_state shape: [batch, 199, 768]
        outputs = self.backbone(input_values, attention_mask=attention_mask).last_hidden_state
        
        # 2. Strategic Pooling (Mean + Std)
        # The mask is resized inside the pooling layer
        features = self.pooling(outputs, attention_mask=attention_mask)
        
        # 3. Predict Emotion
        logits = self.classifier(features)
        return logits

# --- Initialize Model ---
num_classes = len(label2id) # Ensuring label2id from Cell 2 is used
model = VoxSentinelEmotionModel(num_labels=num_classes).to(device)

print("-" * 60)
print(f"✅ Cell 3: Fixed Multi-Scale Architecture Initialized.")
print(f"🔹 Strategy: Attention Mask Interpolation (64000 -> {199 if '199' in str(model) else 'Dynamic'})")
print(f"🔹 Device: {device}")
print("-" * 60)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


------------------------------------------------------------
✅ Cell 3: Fixed Multi-Scale Architecture Initialized.
🔹 Strategy: Attention Mask Interpolation (64000 -> Dynamic)
🔹 Device: cuda
------------------------------------------------------------


In [4]:
# Cell 4: Training Hyperparameters, Loss Function, and Optimizer Setup
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. ⚖️ Calculate Class Weights for Imbalanced Data
# Helps the model focus on underrepresented emotions like 'surprise'
train_emotions = master_df[master_df['split'] == 'train']['emotion']
emotion_counts = train_emotions.value_counts().to_dict()
counts = [emotion_counts[label] for label in label_list]
total = sum(counts)

# Weight Formula: total / (num_classes * class_count)
weights = [total / (len(label_list) * c) for c in counts]
class_weights = torch.FloatTensor(weights).to(device)

# 2. 🎯 Define Loss Function with Weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 3. 🚀 Layer-wise Learning Rate Decay (LLRD)
# Protecting the backbone with 4e-6, while the head learns at 4e-5
optimizer = optim.AdamW([
    {"params": model.backbone.feature_extractor.parameters(), "lr": 0}, 
    {"params": model.backbone.encoder.parameters(), "lr": 4e-6}, 
    {"params": model.pooling.parameters(), "lr": 4e-5},
    {"params": model.classifier.parameters(), "lr": 4e-5},
], weight_decay=0.01)

# 4. 📈 Dynamic Scheduler (Plateau)
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='max',            # Monitoring Validation Accuracy
    factor=0.5, 
    patience=4, 
    min_lr=1e-7
)

# 5. 📦 Batch Size & DataLoader Confirmation
# Ensuring the 32-batch size is locked in for RTX 5080 optimization
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True, 
    drop_last=True
)

dev_loader = DataLoader(
    dev_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=4, 
    pin_memory=True
)

# 6. 🛑 Training Control
EARLY_STOP_PATIENCE = 10
MAX_EPOCHS = 100

print("-" * 60)
print("🚀 VOXSENTINEL HYPERPARAMETERS & LOADERS LOCKED")
print("-" * 60)
print(f"🔹 Configured Batch Size: {BATCH_SIZE}")
print(f"🔹 Train Batches       : {len(train_loader)}")
print(f"🔹 Loss Weights        : {dict(zip(label_list, np.round(weights, 2)))}")
print(f"🔹 Backbone LR         : 4e-6")
print(f"🔹 Head LR             : 4e-5")
print("-" * 60)

------------------------------------------------------------
🚀 VOXSENTINEL HYPERPARAMETERS & LOADERS LOCKED
------------------------------------------------------------
🔹 Configured Batch Size: 32
🔹 Train Batches       : 346
🔹 Loss Weights        : {'angry': np.float64(0.92), 'disgust': np.float64(1.02), 'fear': np.float64(1.02), 'happy': np.float64(0.85), 'neutral': np.float64(0.72), 'sad': np.float64(0.97), 'surprise': np.float64(2.8)}
🔹 Backbone LR         : 4e-6
🔹 Head LR             : 4e-5
------------------------------------------------------------


In [5]:
# Cell 5: Training Loop with Evaluation and Checkpointing
import os
import csv
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

# 1. 📂 Setup & CSV Logging
MODEL_DIR = "./model"
LOG_FILE = os.path.join(MODEL_DIR, "emotion_training_log.csv")
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "dev_acc", "dev_f1", "is_best"])

def save_metrics(epoch, loss, acc, f1, is_best):
    with open(LOG_FILE, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{loss:.6f}", f"{acc:.4f}", f"{f1:.4f}", "YES" if is_best else "NO"])

# 2. 🔍 Evaluation (Refined for Multi-class)
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", leave=False):
            inputs = batch["input_values"].to(device)
            masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(inputs, attention_mask=masks)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro') # 🌟 Macro-F1 handles imbalance
    return acc, f1

# 3. 🔥 Warm Start Logic (Similar to your Deepfake script)
checkpoint_path = os.path.join(MODEL_DIR, "best_emotion_model.pth")
best_f1 = 0.0 

if os.path.exists(checkpoint_path):
    print(f"📦 Found existing checkpoint: {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        # Calibration
        print("🔍 Calibrating baseline metrics...")
        _, best_f1 = evaluate(model, dev_loader, device)
        print(f"✅ Baseline Macro-F1 aligned to: {best_f1:.4f}")
    except Exception as e:
        print(f"⚠️ Load failed: {e}. Starting fresh.")

# 4. 🏁 Training Loop
counter = 0
for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for batch in pbar:
        optimizer.zero_grad()
        inputs, masks, labels = batch["input_values"].to(device), batch["attention_mask"].to(device), batch["label"].to(device)
        
        logits = model(inputs, attention_mask=masks)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[-1]['lr']:.2e}")

    # Validation
    dev_acc, dev_f1 = evaluate(model, dev_loader, device)
    avg_loss = total_loss / len(train_loader)
    
    # Scheduler: Step based on Macro-F1
    scheduler.step(dev_f1)
    
    is_best = dev_f1 > best_f1
    save_metrics(epoch, avg_loss, dev_acc, dev_f1, is_best)
    
    if is_best:
        best_f1 = dev_f1
        counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_map': label2id,
            'f1': dev_f1
        }, checkpoint_path)
        print(f"🥇 New Best Macro-F1: {dev_f1:.4f} -> Saved!")
    else:
        counter += 1
        print(f"⏱ No improvement. [{counter}/{EARLY_STOP_PATIENCE}]")
    
    if counter >= EARLY_STOP_PATIENCE:
        print(f"🛑 Early Stopping. Best F1: {best_f1:.4f}")
        break

print("\n✨ Training session complete.")

Epoch 0: 100%|██████████| 346/346 [01:54<00:00,  3.03it/s, loss=1.4634, lr=4.00e-05]


🥇 New Best Macro-F1: 0.5062 -> Saved!


Epoch 1: 100%|██████████| 346/346 [01:53<00:00,  3.04it/s, loss=1.1644, lr=4.00e-05]


🥇 New Best Macro-F1: 0.6077 -> Saved!


Epoch 2: 100%|██████████| 346/346 [01:54<00:00,  3.03it/s, loss=0.8318, lr=4.00e-05]


...

⏱ No improvement. [8/10]


Epoch 45: 100%|██████████| 346/346 [01:54<00:00,  3.03it/s, loss=0.1181, lr=5.00e-06]


⏱ No improvement. [9/10]


Epoch 46: 100%|██████████| 346/346 [01:54<00:00,  3.03it/s, loss=0.1313, lr=5.00e-06]
                                                           

⏱ No improvement. [10/10]
🛑 Early Stopping. Best F1: 0.7755

✨ Training session complete.


In [6]:
# Cell 6: Final Testing and Comprehensive Reporting
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. 📥 Load the Best Model
checkpoint_path = os.path.join(MODEL_DIR, "best_emotion_model.pth")
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Missing {checkpoint_path}! Did you finish training?")

print(f"📦 Loading best model weights from {checkpoint_path}...")
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# 2. 🧪 Testing Function with Dataset Breakdown
def run_final_test(model, test_df, processor, label_map, device):
    # Prepare the Test Dataset (No augmentation!)
    test_dataset = VoxSentinelDataset(test_df, processor, label_map, augment=False)
    test_loader = DataLoader(test_dataset, batch_size=32, num_workers=4, pin_memory=True)
    
    all_preds = []
    all_labels = []
    
    print(f"🔍 Running inference on {len(test_df)} samples...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            inputs = batch["input_values"].to(device)
            masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(inputs, attention_mask=masks)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Add predictions back to the test dataframe for grouped analysis
    test_df['pred'] = all_preds
    test_df['label_id'] = all_labels
    return test_df

# 3. 🚀 Execute Test
results_df = run_final_test(model, test_df.copy(), processor, label2id, device)

# 4. 📊 Report Generation
print("\n" + "="*60)
print("              VOXSENTINEL GLOBAL PERFORMANCE")
print("="*60)

# Global Accuracy
overall_acc = accuracy_score(results_df['label_id'], results_df['pred'])
print(f"OVERALL TEST ACCURACY: {overall_acc:.4f}")

print("\n" + "-"*60)
print("          PER-DATASET ACCURACY BREAKDOWN")
print("-"*60)

# Group by 'dataset' and calculate accuracy for each
datasets = results_df['dataset'].unique()
for ds_name in datasets:
    sub_df = results_df[results_df['dataset'] == ds_name]
    acc = accuracy_score(sub_df['label_id'], sub_df['pred'])
    count = len(sub_df)
    print(f"📍 {ds_name:<12} | Accuracy: {acc:.4f} | Samples: {count}")

print("-" * 60)
print("\n[Detail] Classification Report (Global):")
# Get class names from label2id
target_names = [k for k, v in sorted(label2id.items(), key=lambda item: item[1])]
print(classification_report(results_df['label_id'], results_df['pred'], target_names=target_names))

# Optional: Save results to CSV for further manual forensic analysis
results_df.to_csv(os.path.join(MODEL_DIR, "test_predictions_detailed.csv"), index=False)
print(f"✅ Detailed predictions saved to {MODEL_DIR}/test_predictions_detailed.csv")

📦 Loading best model weights from ./model/best_emotion_model.pth...
🔍 Running inference on 1631 samples...


Testing: 100%|██████████| 51/51 [00:05<00:00,  9.57it/s]


              VOXSENTINEL GLOBAL PERFORMANCE
OVERALL TEST ACCURACY: 0.6971

------------------------------------------------------------
          PER-DATASET ACCURACY BREAKDOWN
------------------------------------------------------------
📍 crema        | Accuracy: 0.7490 | Samples: 745
📍 ravdess      | Accuracy: 0.8880 | Samples: 125
📍 tess         | Accuracy: 1.0000 | Samples: 260
📍 meld         | Accuracy: 0.4152 | Samples: 501
------------------------------------------------------------

[Detail] Classification Report (Global):
              precision    recall  f1-score   support

       angry       0.68      0.77      0.73       248
     disgust       0.78      0.76      0.77       198
        fear       0.78      0.70      0.74       196
       happy       0.75      0.62      0.68       250
     neutral       0.69      0.72      0.70       423
         sad       0.65      0.64      0.65       227
    surprise       0.48      0.55      0.51        89

    accuracy               